# Self-Correcting Code Generator | Reflection/Self-Correction

In [22]:
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Literal
from typing_extensions import NotRequired
from langchain_core.globals import set_llm_cache
from langchain_community.cache import SQLiteCache
from helper import plot_mermaid, stream_invoke

In [14]:
# Setup Response caching
set_llm_cache(SQLiteCache(database_path=".langchain_cache.db"))

In [15]:
model = ChatOpenAI(model="gpt-4o")

In [16]:
class ReflectionState(TypedDict):
    input: str
    draft: NotRequired[str]
    critique: NotRequired[str]
    iteration: NotRequired[int]
    final_output: NotRequired[str]

In [17]:
MAX_ITERATIONS = 3

In [23]:
# Generator: produce or refine content
def generate(state: ReflectionState) -> dict:
    if state.get("critique"):
        prompt = (
            f"Revise your previous draft based on this feedback.\n\n"
            f"Original request: {state['input']}\n"
            f"Previous draft:\n{state['draft']}\n"
            f"Feedback:\n{state['critique']}"
        )
    else:
        prompt = f"Write a Python function for: {state['input']}"

    response = model.invoke(prompt)
    iteration = state.get("iteration", 0) + 1
    return {"draft": response.content, "iteration": iteration}

# Critic: evaluate and provide feedback
def reflect(state: ReflectionState) -> dict:
    response = model.invoke(
        f"Review this code for correctness, edge cases, and best practices. "
        f"If it's good, respond with exactly 'APPROVED'. "
        f"Otherwise, provide specific feedback.\n\n{state['draft']}"
    )
    return {"critique": response.content}

# Routing: continue or finish (exact match to avoid false positives)
def should_continue(state: ReflectionState) -> Literal['finalize', 'generate']:
    critique = state.get("critique", "").strip()
    iteration = state.get("iteration", 0)
    if critique == "APPROVED" or iteration >= MAX_ITERATIONS:
        return "finalize"
    return "generate"

def finalize(state: ReflectionState) -> dict:
    return {"final_output": state["draft"]}

In [24]:
# Build graph
graph = StateGraph(ReflectionState)
graph.add_node("generate", generate)
graph.add_node("reflect", reflect)
graph.add_node("finalize", finalize)

graph.add_edge(START, "generate")
graph.add_edge("generate", "reflect")
graph.add_conditional_edges("reflect", should_continue)
graph.add_edge("finalize", END)

reflection_agent = graph.compile()

In [25]:
plot_mermaid(reflection_agent)

```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	generate(generate)
	reflect(reflect)
	finalize(finalize)
	__end__([<p>__end__</p>]):::last
	__start__ --> generate;
	generate --> reflect;
	reflect -.-> finalize;
	reflect -.-> generate;
	finalize --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

In [26]:
result = reflection_agent.invoke({
    "input": "a function that finds all prime numbers up to N using the Sieve of Eratosthenes"
})
print(result["final_output"])

Certainly! I've incorporated the recommended improvements, focusing on input validation and maintaining clarity and performance:

```python
def sieve_of_eratosthenes(n):
    """
    Return a list of all prime numbers up to n using the Sieve of Eratosthenes algorithm.

    Parameters:
    n (int): The upper limit for finding prime numbers.

    Returns:
    list: A list of prime numbers up to n.
    """
    # Input validation: Ensure n is an integer
    if not isinstance(n, int):
        raise ValueError("The input must be an integer.")
    
    if n < 2:
        # If n is less than 2, there are no prime numbers
        return []

    # Initialize a boolean list to track prime status
    is_prime = [True] * (n + 1)
    p = 2

    # Iterate over numbers from 2 to square root of n
    while p * p <= n:
        # If is_prime[p] is True, p is a prime number
        if is_prime[p]:
            # Mark all multiples of p starting from p^2 as non-prime
            for i in range(p * p, n + 1, p

In [27]:
stream_invoke(reflection_agent, {
    "input": "a function that finds all prime numbers up to N using the Sieve of Eratosthenes"
})


────────────────────────────────────────────────────────────────────────────────
  STREAMING EXECUTION
────────────────────────────────────────────────────────────────────────────────

────────────────────────────────────────────────────────────────────────────────
  EXECUTION COMPLETE
────────────────────────────────────────────────────────────────────────────────



{'input': 'a function that finds all prime numbers up to N using the Sieve of Eratosthenes',
 'draft': 'Certainly! I\'ve incorporated the recommended improvements, focusing on input validation and maintaining clarity and performance:\n\n```python\ndef sieve_of_eratosthenes(n):\n    """\n    Return a list of all prime numbers up to n using the Sieve of Eratosthenes algorithm.\n\n    Parameters:\n    n (int): The upper limit for finding prime numbers.\n\n    Returns:\n    list: A list of prime numbers up to n.\n    """\n    # Input validation: Ensure n is an integer\n    if not isinstance(n, int):\n        raise ValueError("The input must be an integer.")\n    \n    if n < 2:\n        # If n is less than 2, there are no prime numbers\n        return []\n\n    # Initialize a boolean list to track prime status\n    is_prime = [True] * (n + 1)\n    p = 2\n\n    # Iterate over numbers from 2 to square root of n\n    while p * p <= n:\n        # If is_prime[p] is True, p is a prime number\n  